# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

# Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

---

### Task 1: Prior Belief Boundaries

#### Analytical Expected Value
For a Beta distribution $\Theta \sim \text{Beta}(\alpha, \beta)$ with parameters $\alpha = 8$ and $\beta = 1.5$:

$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta} = \frac{8}{8 + 1.5} = \frac{8}{9.5} = \frac{16}{19} \approx 0.8421$$

#### Physical Justification
The $\text{Beta}(8, 1.5)$ distribution serves as an ideal initial prior for an engineering component assumed to be healthy:
* **Asymmetric Skewness:** Most of the probability mass is concentrated heavily near $\theta = 1.0$, which reflects engineering confidence that a newly deployed or baseline component is undamaged.
* **Bounded Support:** It naturally respects the physical bounds $\theta \in (0, 1]$, assigning zero probability to unphysical negative stiffness or efficiency greater than 100%.
* **Uncertainty Buffer:** Unlike a deterministic point mass ($\theta = 1.0$), it acknowledges manufacturing variances and leaves non-zero probability mass across lower stiffness regions, allowing incoming sensor data to shift belief if damage occurs.

---

### Task 2: Structural Likelihood Formulation

#### Single Sensor Measurement
The physical forward model for a sensor reading $y_k$ at step $k$ given the stiffness parameter $\theta$ is:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

Taking the natural logarithm yields:

$$\ln(y_k) = \ln(\theta \cdot K_{\text{nominal}}) + \epsilon_k$$

Since $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$, the term $\ln(y_k)$ follows a Gaussian distribution: $\ln(y_k) \sim \mathscr{N}(\ln(\theta \cdot K_{\text{nominal}}), \sigma^2)$. Applying the transformation rule for probability densities $f_{Y_k}(y_k) = f_{\ln Y_k}(\ln y_k) \cdot \left| \frac{d}{dy_k} \ln y_k \right|$ yields the log-normal likelihood for a single measurement $y_k$:

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp \left( -\frac{\left( \ln y_k - \ln(\theta \cdot K_{\text{nominal}}) \right)^2}{2\sigma^2} \right)$$

#### Joint Likelihood Function
Assuming conditionally independent measurement errors across inspection steps given $\Theta = \theta$, the joint likelihood function for the continuous measurement history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} L(y_i \mid \theta) = \left( \frac{1}{\sigma \sqrt{2\pi}} \right)^k \left( \prod_{i=1}^k \frac{1}{y_i} \right) \exp \left( -\frac{1}{2\sigma^2} \sum_{i=1}^{k} \left( \ln y_i - \ln(\theta \cdot K_{\text{nominal}}) \right)^2 \right)$$

---

### Task 3: Mathematical Formulation of the Non-Conjugate Grid Update

#### Absence of Analytical Closed-Form
A closed-form analytical posterior does not exist because the Beta prior (algebraic functional form $\theta^{\alpha-1}(1-\theta)^{\beta-1}$) is **non-conjugate** to the log-normal measurement likelihood (where $\theta$ appears inside a logarithmic term within an exponential function). Their product cannot be integrated analytically to express the normalizing marginal likelihood in closed form.

#### Recursive Bayesian Formulation
Applying Bayes' theorem recursively, the posterior density at step $k$ satisfies:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Including the normalization constant $Z_k$:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{0}^{1} L(y_k \mid s) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(s \mid \mathbf{y}^{(k-1)}) \, ds}$$

---

### Task 4: Running Point Estimators

Across the bounded domain $(0, 1]$, the running point estimates at step $k$ are defined via the following definite integrals:

#### 1. Running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
Under minimum mean squared error (MMSE) loss:

$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb{E}\left[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}\right] = \int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$

#### 2. Running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)
Under zero-one loss:

$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

---

### Task 5: Algorithmic Grid Approximation and Normalization

1. **Grid Discretization:** Construct an array of $M$ equally spaced points over $[\theta_{\min}, \theta_{\max}] = [0.001, 1.0]$:
   $$\theta_m = \theta_{\min} + (m-1)\Delta\theta, \qquad \Delta\theta = \frac{\theta_{\max} - \theta_{\min}}{M-1}, \quad m = 1, \dots, M$$
2. **Prior Discretization & Normalization:** Compute $P_0(\theta_m) = \text{Beta}(\theta_m; 8, 1.5)$ and normalize using the composite trapezoidal rule:
   $$Z_0 = \int_{\theta_{\min}}^{\theta_{\max}} P_0(\theta) d\theta \approx \sum_{m=1}^{M-1} \frac{P_0(\theta_m) + P_0(\theta_{m+1})}{2} \Delta\theta, \qquad P_0(\theta_m) \leftarrow \frac{P_0(\theta_m)}{Z_0}$$
3. **Sequential Updating at Step $k$:**
   * Evaluate Likelihood: $L_k(\theta_m) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp \left( -\frac{(\ln y_k - \ln(\theta_m \cdot K_{\text{nominal}}))^2}{2\sigma^2} \right)$
   * Compute Unnormalized Array: $\tilde{P}_k(\theta_m) = P_{k-1}(\theta_m) \times L_k(\theta_m)$
   * Compute Normalizing Integral:
     $$Z_k = \sum_{m=1}^{M-1} \frac{\tilde{P}_k(\theta_m) + \tilde{P}_k(\theta_{m+1})}{2} \Delta\theta$$
   * Normalize Density Array: $P_k(\theta_m) = \frac{\tilde{P}_k(\theta_m)}{Z_k}$
4. **Point Estimation Evaluation:**
   * **Bayes Estimate:** $\widehat{\theta}_{\mathrm{Bayes}}^{(k)} \approx \text{trapezoid}(\boldsymbol{\theta} \odot \mathbf{P}_k, \boldsymbol{\theta})$
   * **MAP Estimate:** $\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \theta_{m^*}, \quad \text{where } m^* = \arg\max_m P_k(\theta_m)$

In [1]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set random seed for reproducible noisy sensor data
np.random.seed(42)

# --- Physical & Model Parameters ---
K_nominal = 50.0       # Baseline healthy stiffness (kN/mm)
sigma = 0.15           # Log-space measurement noise standard deviation
theta_true = 0.68      # Hidden true remaining stiffness (68% health)
n_steps = 15           # Total inspection steps

# Beta prior parameters
alpha_0, beta_0 = 8.0, 1.5

# --- Task 5: Grid Discretization ---
M = 1000               # Grid resolution
theta_grid = np.linspace(0.001, 1.0, M)

# --- Task 1: Prior Initialization ---
prior_pdf = stats.beta.pdf(theta_grid, alpha_0, beta_0)
# Normalize prior using composite trapezoidal rule
prior_pdf /= np.trapezoid(prior_pdf, theta_grid)

# --- Task 6: Data Generation & Tracking ---
# Simulate log-normal sensor stream: y_k = theta_true * K_nominal * exp(epsilon_k)
sensor_readings = theta_true * K_nominal * np.exp(np.random.normal(0, sigma, size=n_steps))

# Matrices/Arrays to store posterior history and point estimates
posterior_history = np.zeros((n_steps + 1, M))
posterior_history[0, :] = prior_pdf

bayes_estimates = np.zeros(n_steps + 1)
map_estimates = np.zeros(n_steps + 1)

# Step 0 Estimators
bayes_estimates[0] = np.trapezoid(theta_grid * prior_pdf, theta_grid)
map_estimates[0] = theta_grid[np.argmax(prior_pdf)]

# Sequential Updating Loop
current_posterior = prior_pdf.copy()

for k in range(1, n_steps + 1):
    y_k = sensor_readings[k - 1]

    # Task 2: Log-normal Likelihood for single measurement y_k
    mu_log = np.log(theta_grid * K_nominal)
    likelihood = (1.0 / (y_k * sigma * np.sqrt(2 * np.pi))) * np.exp(
        -0.5 * ((np.log(y_k) - mu_log) / sigma) ** 2
    )

    # Task 3 & 5: Unnormalized Update & Numerical Trapezoidal Normalization
    unnormalized_posterior = current_posterior * likelihood
    Z_k = np.trapezoid(unnormalized_posterior, theta_grid)
    current_posterior = unnormalized_posterior / Z_k

    # Save posterior
    posterior_history[k, :] = current_posterior

    # Task 4 & 5: Numerical Estimators
    bayes_estimates[k] = np.trapezoid(theta_grid * current_posterior, theta_grid)
    map_estimates[k] = theta_grid[np.argmax(current_posterior)]

# --- Plotting via Plotly ---

# Plot 1: Posterior Density Curves at Milestones k in {0, 1, 2, 5, 10, 15}
milestones = [0, 1, 2, 5, 10, 15]
fig_posterior = go.Figure()

colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3']

for idx, k in enumerate(milestones):
    fig_posterior.add_trace(go.Scatter(
        x=theta_grid,
        y=posterior_history[k],
        mode='lines',
        name=f'Step k={k}',
        line=dict(color=colors[idx], width=2.5)
    ))

fig_posterior.add_vline(
    x=theta_true,
    line_dash="dash",
    line_color="red",
    annotation_text="True θ = 0.68",
    annotation_position="top right"
)

fig_posterior.update_layout(
    title="<b>Sequential Bayesian Density Evolution via Bounded Grid Updates</b>",
    xaxis_title="Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density",
    template="plotly_white",
    hovermode="x unified",
    width=900,
    height=500
)

fig_posterior.show()

# Plot 2: Convergence Timeline Chart (Bayes vs MAP vs True Line)
fig_tracking = go.Figure()

steps = np.arange(n_steps + 1)

fig_tracking.add_trace(go.Scatter(
    x=steps, y=bayes_estimates,
    mode='lines+markers',
    name='Posterior Mean (Bayes)',
    line=dict(color='#1f77b4', width=2),
    marker=dict(size=6)
))

fig_tracking.add_trace(go.Scatter(
    x=steps, y=map_estimates,
    mode='lines+markers',
    name='MAP Estimator',
    line=dict(color='#ff7f0e', width=2, dash='dot'),
    marker=dict(size=6)
))

fig_tracking.add_trace(go.Scatter(
    x=[0, n_steps], y=[theta_true, theta_true],
    mode='lines',
    name='True Stiffness (θ_true = 0.68)',
    line=dict(color='red', width=2, dash='dash')
))

fig_tracking.update_layout(
    title="<b>Structural Health Monitoring: Parameter Estimator Convergence Timeline</b>",
    xaxis_title="Inspection Step (k)",
    yaxis_title="Estimated Stiffness Efficiency Factor (θ)",
    template="plotly_white",
    xaxis=dict(tickmode='linear', tick0=0, dtick=1),
    hovermode="x unified",
    width=900,
    height=500
)

fig_tracking.show()

### Convergence & Structural Safety Analysis

1. **Prior Resistance vs. Data Convergence Speed:** The initial prior $\text{Beta}(8, 1.5)$ biases initial estimates toward a healthy state ($\mathbb{E}[\Theta^{(0)}] \approx 0.842$). However, as noisy measurements arrive, the likelihood product dominates the posterior update. Within **3 to 4 sensor readings**, the system overcomes the prior bias and stabilizes near the true damaged state ($\theta_{\text{true}} = 0.68$). By step $k = 5$, both $\widehat{\theta}_{\mathrm{Bayes}}$ and $\widehat{\theta}_{\mathrm{MAP}}$ closely mirror $0.68$.

2. **Uncertainty Attenuation & Safety Thresholds:** As the inspection count increases from $k = 0$ to $k = 15$, the variance (width) of the posterior density shrinks dramatically.
   * **At $k = 0$**, probability mass is spread widely across $[0.5, 1.0]$.
   * **At $k = 15$**, the density curve forms a narrow peak around $0.68$.
   * **Safety Threshold Implication:** In structural safety monitoring, automated triggers evaluate tail probabilities (e.g., $P(\Theta < 0.70) \ge 0.95$). As likelihood information concentrates the distribution, the narrow variance allows engineers to trigger structural alerts with statistical confidence while minimizing false alarms.